In [6]:
import rdflib
from rdflib.plugins.sparql import prepareQuery
from tabulate import tabulate

In [7]:
filename = "ABox.ttl"

In [8]:
text1 = '''CQ_14.1
Return the characteristics of all agents that are people.
'''

query1 = '''
PREFIX obj: <https://w3id.org/dharc/ontology/chad-ap/object/development/14/schema/>

SELECT ?full_name ?f_name ?l_name ?country ?language ?gender ?profession ?date_birth ?date_death (GROUP_CONCAT(DISTINCT ?link; separator="; ") AS ?links)
WHERE {
    ?agent a obj:Person .
    OPTIONAL { ?agent obj:isIdentifiedBy [ a obj:Appellation; obj:hasType obj:full-name; obj:hasSymbolicContent ?full_name ] }
    OPTIONAL { ?agent obj:isIdentifiedBy [ a obj:Appellation; obj:hasType obj:first-name; obj:hasSymbolicContent ?f_name ] }
    OPTIONAL { ?agent obj:isIdentifiedBy [ a obj:Appellation; obj:hasType obj:last-name; obj:hasSymbolicContent ?l_name ] }
    OPTIONAL { ?agent obj:hasResidenceIn/obj:isIdentifiedBy/obj:hasSymbolicContent ?country }
    OPTIONAL { ?agent obj:isReferredToBy [ a obj:LinguisticObject; obj:hasSymbolicContent ?profession ] }
    OPTIONAL { 
    ?aa1 obj:assignsAttributeTo ?agent ; obj:assignsPropertyOfType obj:gender ; obj:assigns ?g_val .
    BIND(STR(?g_val) AS ?gender)
    }
    OPTIONAL { ?aa2 obj:assignsAttributeTo ?agent ; obj:assignsPropertyOfType obj:language ; obj:assigns ?language }
    OPTIONAL { 
    ?agent obj:wasPresentAt [ a obj:Event; obj:hasType obj:birth; obj:hasTimeSpan/obj:hasStartTime ?b_dt ] 
    BIND(STRBEFORE(STR(?b_dt), "T") AS ?date_birth)
    }
    OPTIONAL { 
        ?agent obj:wasPresentAt [ a obj:Event; obj:hasType obj:death; obj:hasTimeSpan/obj:hasStartTime ?d_dt ] 
        BIND(STRBEFORE(STR(?d_dt), "T") AS ?date_death)
    }
    OPTIONAL { ?agent obj:isIdentifiedBy [ a obj:Identifier; obj:hasType obj:uri; obj:hasSymbolicContent ?link ] }
}
GROUP BY ?full_name ?f_name ?l_name ?country ?language ?gender ?profession ?date_birth ?date_death
'''

In [9]:
text2 = '''CQ_14.2
Return the characteristics of all agents that are organisations.
'''

query2 = '''
PREFIX obj: <https://w3id.org/dharc/ontology/chad-ap/object/development/14/schema/>

SELECT ?name ?country ?s_date ?e_date (GROUP_CONCAT(DISTINCT ?link; separator="; ") AS ?links)
WHERE {
  ?org a obj:Organization .
  OPTIONAL { ?org obj:isIdentifiedBy [ a obj:Appellation; obj:hasType obj:full-name; obj:hasSymbolicContent ?name ] }
  OPTIONAL { ?org obj:hasResidenceIn/obj:isIdentifiedBy/obj:hasSymbolicContent ?country }
  OPTIONAL { 
    ?org obj:wasPresentAt [ a obj:Event; obj:hasType obj:foundation; obj:hasTimeSpan/obj:hasStartTime ?sd ] 
    BIND(STRBEFORE(STR(?sd), "T") AS ?s_date)
  }
  OPTIONAL { 
    ?org obj:wasPresentAt [ a obj:Event; obj:hasType obj:dissolution; obj:hasTimeSpan/obj:hasStartTime ?ed ] 
    BIND(STRBEFORE(STR(?ed), "T") AS ?e_date)
  }
  OPTIONAL { ?org obj:isIdentifiedBy [ a obj:Identifier; obj:hasType obj:uri; obj:hasSymbolicContent ?link ] }
}
GROUP BY ?name ?country ?s_date ?e_date
'''

In [10]:
queries = [(text1, query1),
            (text2, query2),
           ]

g = rdflib.ConjunctiveGraph()
g.parse(filename, format="turtle", encoding="utf-8")

for query in queries:
    q = prepareQuery(query[1])
    results = g.query(q)
    print(query[0])
    table = []
    for row in results:
        table.append([row[var] for var in results.vars])
    print(tabulate(table, headers=results.vars, tablefmt="psql"))

CQ_14.1
Return the characteristics of all agents that are people.

+-------------+----------+----------+-------------------------------------------------+-------------------------------------------+---------------------------------------------------------------------------+-------------------------------------------------------------------------------------+--------------+--------------+----------------------------------------------------------------------------------------------------------+
| full_name   | f_name   | l_name   | country                                         | language                                  | gender                                                                    | profession                                                                          | date_birth   | date_death   | links                                                                                                    |
|-------------+----------+----------+----------------------------------